In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
from abc import ABC, abstractmethod
from pydantic import BaseModel, computed_field
from functools import cached_property
from typing import List, Tuple, Optional, Any, ClassVar, Union
from datasets import load_from_disk


class GenModel(BaseModel, ABC):
    base_model_id: ClassVar[str]

    checkpoint_path: str
    timestamp: Optional[int] = None
    steps: Optional[int] = None
    _model_and_tokenizer: Optional[Tuple[Any, Any]] = None

    def _load_model_and_tokenizer(self) -> Tuple[Any, Any]:
        if self._model_and_tokenizer is None:
            model, tokenizer = self.load_model(self.checkpoint_path)
            self._model_and_tokenizer = (model, tokenizer)
        return self._model_and_tokenizer

    @computed_field
    @cached_property
    def tokenizer(self) -> Any:
        _, tokenizer = self._load_model_and_tokenizer()
        return tokenizer

    @computed_field
    @cached_property
    def gen_model(self) -> Any:
        model, _ = self._load_model_and_tokenizer()
        return model

    @staticmethod
    def load_finetuning_dataset(path, tokenizer, token_func=None, chat=True):
        dataset = load_from_disk(path)

        if token_func is not None:
            dataset = dataset.map(token_func)

        if chat:

            def format_chat_data(example):
                convos = example["text"]
                texts = [
                    tokenizer.apply_chat_template(
                        convo,
                        add_generation_prompt=False,
                        tokenize=False,
                        return_tensors="pt",
                    )
                    for convo in convos
                ]
                return {"text": texts}

            dataset = dataset.map(format_chat_data, batched=True)
        else:

            dataset = dataset.map(
                lambda samples: tokenizer(
                    samples["text"],
                    padding="max_length",
                    truncation=True,
                    max_length=512,
                    add_special_tokens=True,
                ),
            ).shuffle()

        # print first sample tokenized

        return dataset

    @classmethod
    @abstractmethod
    def train_model(
        cls, dataset_path: str, callbacks=None, **kwargs
    ) -> List["GenModel"]:
        """
        Train the model and return a list of fine-tuned checkpoints.

        Args:
            dataset_path: Path to the dataset
            callbacks: Optional list of callbacks to use during training
            **kwargs: Additional training arguments
        """
        pass

    @abstractmethod
    def load_model(self, path: str) -> Any:
        """
        Load the checkpoint from file system.
        """
        pass

    @abstractmethod
    def generate(self, prompt: str, gen_args: dict) -> str:
        """
        Run inference on the given prompt using the fine-tuned model.
        """
        pass

    @abstractmethod
    def resume_training(self, steps: int, callbacks=None) -> List["GenModel"]:
        """
        Resume training from self.

        Args:
            steps: Number of steps to train for
            callbacks: Optional list of callbacks to use during training
        """
        pass


In [3]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from typing import Any, Tuple, ClassVar, List
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments

import os
from datetime import datetime


class LLama31GenModel(GenModel):
    base_model_id: ClassVar[str] = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

    @classmethod
    def train_model(
        cls,
        dataset_path: str,
        epochs: int = 2,
        max_seq_length: int = 512,
        learning_rate: float = 2e-5,
        lr_scheduler: str = "cosine",
        gradient_accumulation_steps: int = 1,
        weight_decay: float = 0.01,
        warmup_steps: int = 100,
        lora_rank: int = 64,
        save_steps: int = 500,
        output_dir: str = "finetuning/sft/models",
    ) -> List["LLama31GenModel"]:
        base_model_id = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_id,
            load_in_4bit=True,
            dtype=None,
        )

        # change the padding tokenizer value
        tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
        model.config.pad_token_id = tokenizer.pad_token_id  # updating model config
        tokenizer.padding_side = (
            "right"  # padding to right (otherwise SFTTrainer shows warning)
        )

        # add eos token at the end of the samples

        def add_eos_token(example):
            example["text"] = example["text"] + tokenizer.eos_token
            return example

        dataset = cls.load_finetuning_dataset(
            path=dataset_path, tokenizer=tokenizer, token_func=add_eos_token
        )
        print(dataset[0])

        response_template = "\n->\n"
        collator = DataCollatorForCompletionOnlyLM(
            tokenizer=tokenizer, response_template=response_template
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r=lora_rank,
            lora_alpha=16,
            lora_dropout=0,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "up_proj",
                "down_proj",
                "o_proj",
                "gate_proj",
            ],
            use_rslora=True,
            use_gradient_checkpointing="unsloth",
        )

        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset,
            tokenizer=tokenizer,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            data_collator=collator,
            args=TrainingArguments(
                learning_rate=learning_rate,
                lr_scheduler_type=lr_scheduler,
                per_device_train_batch_size=2,
                gradient_accumulation_steps=gradient_accumulation_steps,
                num_train_epochs=epochs,
                fp16=not is_bfloat16_supported(),
                bf16=is_bfloat16_supported(),
                logging_steps=1,
                optim="adamw_8bit",
                weight_decay=weight_decay,
                warmup_steps=warmup_steps,
                output_dir=output_dir,
                save_steps=save_steps,
            ),
        )

        trainer.train()

        timestamp = int(datetime.now().timestamp())
        checkpoint_steps = []
        checkpoint_dir = os.path.join(output_dir)

        if os.path.exists(checkpoint_dir):
            for folder in os.listdir(checkpoint_dir):
                if folder.startswith("checkpoint-"):
                    step = int(folder.split("-")[1])
                    checkpoint_steps.append(step)

        checkpoint_models = []
        for step in checkpoint_steps:
            checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint-{step}")
            model_instance = cls(
                checkpoint_path=checkpoint_path, timestamp=timestamp, steps=step
            )
            checkpoint_models.append(model_instance)

        return checkpoint_models

    def resume_training(self, epochs):
        raise NotImplementedError

    def load_model(self, checkpoint_path: str) -> Tuple[Any, Any]:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=checkpoint_path,
            max_seq_length=512,  # temp fix
            load_in_4bit=True,
            dtype=None,
        )

        tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
        model.config.pad_token_id = tokenizer.pad_token_id  # updating model config
        tokenizer.padding_side = (
            "right"  # padding to right (otherwise SFTTrainer shows warning)
        )

        model = FastLanguageModel.for_inference(model)

        return model, tokenizer

    def export_gguf(self, checkpoint, quantization_method="fp16"):
        model_folder = f"finetuning/sft/models/{self.base_model_id.split('/')[-1]}/{self.timestamp}"
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=f"{model_folder}/checkpoint-{checkpoint}",
            max_seq_length=self.fine_tuning_arguments.max_seq_length,
            load_in_4bit=True,
            dtype=None,
        )

        tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_0|>"})
        model.config.pad_token_id = tokenizer.pad_token_id
        tokenizer.padding_side = "right"

        model.save_pretrained_gguf("gguf", tokenizer, quantization_method)

    def generate(self, prompt: str, gen_args: dict) -> str:
        input_ids = self.tokenizer(
            prompt, return_tensors="pt", padding=True, truncation=True
        ).input_ids.to("cuda")
        attention_mask = self.tokenizer(
            prompt, return_tensors="pt", padding=True, truncation=True
        ).attention_mask.to("cuda")

        output = self.gen_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=gen_args.get("max_length", 512),
            do_sample=gen_args.get("do_sample", True),
            temperature=gen_args.get("temperature", 0.7),
            top_p=gen_args.get("top_p", 0.95),
            top_k=gen_args.get("top_k", 50),
        )
        generated_text = self.tokenizer.decode(
            output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        return generated_text.strip()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer


In [4]:
from pydantic import BaseModel
from typing import List, Optional, Dict, Union


# TODO: check whether attachments and URLS fields are always present in datataset
class Prompt(BaseModel):
    subject: str
    attachments: Optional[bool]
    urls: Optional[bool]


class OutputMessage(BaseModel):
    body: str
    # attachments: Optional[List[str]]
    # urls: Optional[List[str]]


class PromptOutputPair(BaseModel):

    prompt: Prompt
    output_message: OutputMessage


def generate_prompt(
    subject: str, attachments: bool = False, urls: bool = False, sentiment:list = ["neutral"]
) -> str:
    prompt = dict()
    prompt["subject"] = subject
    prompt["urls"] = urls
    prompt["attachments"] = attachments

    if sentiment:

        prompt["sentiment"] = ", ".join(sentiment)

    prompt = "\n".join(f"{k}: {v}" for k, v in prompt.items())

    return prompt


def generate_target_value(body):
    target_value = {"body": body}
    target_value = "\n".join(f"{k}: {v}" for k, v in target_value.items())

    return target_value


def generate_prompt_output_pair(
    body: str,
    subject: str,
    attachments: bool = False,
    urls: bool = False,
    sentiment:List[str] = ["neutral"],
) -> str:
    prompt = generate_prompt(subject, attachments, urls, sentiment=sentiment)
    target_value = generate_target_value(body)
    return f"{prompt}\n->\n{target_value}"


In [5]:
from pydantic import BaseModel
from typing import Optional, List


class MessageGenerator(BaseModel):
    gen_model: GenModel

    def generate_message(
        self,
        subject: str,
        attachments: bool = False,
        sentiment: List[str] = ["neutral"],
        urls: bool = False,
    ):

        prompt = (
            generate_prompt(
                subject=subject, attachments=attachments, urls=urls, sentiment=sentiment
            )
            + "\n->\n"
        )

        gen_args = {
            "max_length": 384,
            "num_return_sequences": 1,
            "top_k": 50,
            "top_p": 0.95,
            "do_sample": True,
            "temperature": 0.9,
        }

        return self.gen_model.generate(
            prompt=prompt,
            gen_args=gen_args,
        )

    def generate_chat_message(
        self,
        subject: str,
        attachments: bool = False,
        sentiment: List[str] = ["neutral"],
        urls: bool = False,
    ):

        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that assists in writing emails.\n\nCompose an email based on the features provided by the user.\n\nAll identifiable information should be replaced by corresponding placeholders:\n\nurls -> <URL>\nattachments -> <ATTACHMENT>\nphone numbers -> <PHONE>\ndates -> <DATE>\norganization  -> <ORG>\nemail address -> <EMAIL>\nperson name -> <PER>\naddress/location -> <LOC>",
            },
            {
                "role": "user",
                "content": f"urls: {urls}\nattachments: {attachments}\nsubject: {subject}",
            },
        ]
        gen_args = {
            "max_length": 512,
            "num_return_sequences": 1,
            "top_k": 50,
            "top_p": 0.95,
            "do_sample": True,
            "temperature": 0.75,
        }
        return self.gen_model.generate(prompt=messages, gen_args=gen_args)


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
model = LLama31GenModel(checkpoint_path="/content/drive/MyDrive/Thesisproject/Models/checkpoint-2104")

mess_gen = MessageGenerator(gen_model=model)
response = mess_gen.generate_message(
    subject="Join Us for an Environmental Hackathon: Solve Real-World Challenges",
    attachments=False,
    sentiment=["neutral"],
    urls=True,
)
print(response)


==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth: Will load /content/drive/MyDrive/Thesisproject/Models/checkpoint-2104 as a legacy tokenizer.
Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


subject: Join Us for an Environmental Hackathon: Solve Real-World Challenges
urls: True
attachments: False
sentiment: neutral
->
body: The <ORG> and <ORG> are hosting a two-day environmental hackathon to engage students and professionals across the university to find innovative, tech-driven solutions to environmental problems. The event will bring together teams of students, researchers, and faculty to work collaboratively to solve real-world problems and issues. Read more at <URL> <<URL>
